# 01 — EDA & Preprocessing
**Author:** Ahmed Deraz (24046227)  
**Project:** Hourly Urban Air Quality Prediction Using Machine Learning  
**Module:** UFCFAS-15-2 Machine Learning  

This notebook performs exploratory data analysis (EDA) on the UrbanAirNet dataset and validates the shared preprocessing pipeline.

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Make sure figures directory exists
os.makedirs('../report/figures', exist_ok=True)

df = pd.read_csv('../data/urban_air_quality.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Basic Statistics

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Descriptive Statistics ---')
df.describe()

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
missing_nonzero = missing[missing > 0]

print(f'Columns with missing values: {len(missing_nonzero)}')
print(missing_nonzero)

if len(missing_nonzero) > 0:
    plt.figure(figsize=(12, 5))
    missing_nonzero.plot(kind='bar', color='coral', edgecolor='black')
    plt.axhline(y=0.4, color='red', linestyle='--', label='40% threshold (drop)')
    plt.title('Proportion of Missing Values per Feature', fontsize=14)
    plt.ylabel('Missing Fraction')
    plt.xlabel('Feature')
    plt.legend()
    plt.tight_layout()
    plt.savefig('../report/figures/missing_values.png', dpi=150)
    plt.show()
    print('Saved: report/figures/missing_values.png')
else:
    print('No missing values found — dataset is complete.')

## 4. Target Variable Distribution (AQI_Target)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(df['AQI_Target'].dropna(), bins=60, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_title('AQI_Target Distribution')
axes[0].set_xlabel('AQI')
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(df['AQI_Target'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('AQI_Target Box Plot')
axes[1].set_ylabel('AQI')

plt.tight_layout()
plt.savefig('../report/figures/aqi_distribution.png', dpi=150)
plt.show()
print('Saved: report/figures/aqi_distribution.png')

print()
print('AQI_Target summary statistics:')
print(df['AQI_Target'].describe())

## 5. AQI Over Time

In [ ]:
# Try to detect a datetime column
datetime_candidates = ['timestamp', 'date', 'datetime', 'time', 'Date', 'Datetime']
dt_col = next((c for c in datetime_candidates if c in df.columns), None)

if dt_col:
    df_time = df.copy()
    df_time[dt_col] = pd.to_datetime(df_time[dt_col])
    df_time = df_time.set_index(dt_col).sort_index()
    daily_avg = df_time['AQI_Target'].resample('D').mean()

    plt.figure(figsize=(14, 4))
    daily_avg.plot(color='darkorange', linewidth=0.8)
    plt.title('Daily Average AQI Over Time')
    plt.ylabel('AQI')
    plt.xlabel('Date')
    plt.tight_layout()
    plt.savefig('../report/figures/aqi_over_time.png', dpi=150)
    plt.show()
    print('Saved: report/figures/aqi_over_time.png')
else:
    print('No datetime column detected. Plotting AQI by row index instead.')
    plt.figure(figsize=(14, 4))
    plt.plot(df['AQI_Target'].values[:5000], color='darkorange', linewidth=0.5)
    plt.title('AQI_Target (first 5000 rows)')
    plt.ylabel('AQI')
    plt.xlabel('Row Index')
    plt.tight_layout()
    plt.savefig('../report/figures/aqi_over_time.png', dpi=150)
    plt.show()

## 6. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=False, linewidths=0.3, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('../report/figures/correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: report/figures/correlation_heatmap.png')

# Top correlations with AQI_Target
print('\nTop 10 features correlated with AQI_Target:')
print(corr['AQI_Target'].drop('AQI_Target').abs().sort_values(ascending=False).head(10))

## 7. Pollutant Distributions

In [ ]:
pollutants = [c for c in ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3'] if c in df.columns]

if pollutants:
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.flatten()
    colors = ['#e63946','#f4a261','#2a9d8f','#264653','#8338ec','#fb8500']
    for i, col in enumerate(pollutants[:6]):
        axes[i].hist(df[col].dropna(), bins=50, color=colors[i], edgecolor='black', alpha=0.8)
        axes[i].set_title(col)
        axes[i].set_xlabel('Concentration')
        axes[i].set_ylabel('Frequency')
    for j in range(len(pollutants), 6):
        axes[j].set_visible(False)
    plt.suptitle('Pollutant Concentration Distributions', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('../report/figures/pollutant_distributions.png', dpi=150)
    plt.show()
    print('Saved: report/figures/pollutant_distributions.png')
else:
    print('Standard pollutant columns not found. Check column names in the dataset.')
    print('Available numeric columns:', list(df.select_dtypes(include=[np.number]).columns))

## 8. Validate Preprocessing Pipeline

In [ ]:
import sys
sys.path.append('../src')
from preprocessing import load_and_preprocess

X_train, X_test, y_train, y_test, feature_names = load_and_preprocess()

print(f'\n✅ Preprocessing complete!')
print(f'   X_train shape : {X_train.shape}')
print(f'   X_test shape  : {X_test.shape}')
print(f'   y_train shape : {y_train.shape}')
print(f'   y_test shape  : {y_test.shape}')
print(f'   Features used : {len(feature_names)}')
print(f'   Feature list  : {feature_names}')
print()
print(f'   AQI_Target train range: [{y_train.min():.2f}, {y_train.max():.2f}]')
print(f'   AQI_Target test range : [{y_test.min():.2f}, {y_test.max():.2f}]')

---
**End of Notebook 01** — EDA complete. Proceed to `02_linear_regression.ipynb`.